In [ ]:
from TrainingFramework import TrainingFramework
from ECGEncoder import ECGEncoder
from TextEncoder import TextEncoder
from SharedMetricSpace import SharedMetricSpace
from ContrastiveLearning import ContrastiveLearning
from ReportDecoder import ReportDecoder

from ECGDataLoader import ECGDataLoader, ECGDataBase
import warnings
warnings.filterwarnings("ignore")

TRAINING_PARAMS = \
{
    'num_epochs': 2,
    'batch_size': 8,
    'caption_loss_weight': 2,
    'contrastive_loss_weight': 1
}
#Load ECG images from MIMIC-IV-data-mini-subset/mimic-iv-ecg-matched-subset/mimic-iv-ecg_complete_300x300_images
ECG_DATA_DIR_MINI_SUBSET_IMAGES = "mimic-iv-ecg_complete_300x300_images"

{'num_epochs': 2, 'batch_size': 8, 'caption_loss_weight': 2, 'contrastive_loss_weight': 1}


In [ ]:
#Instantiate database
edb = ECGDataBase(ecg_data_dir=ECG_DATA_DIR_MINI_SUBSET_IMAGES)
#uncomment to randomly remove N-100 samples from the database:
#edb.random_undersample(100)

In [ ]:
#Subject-wise train val split 
tr_subjects, vl_subjects = edb.train_val_split()
print(f"tr_subjects: {len(tr_subjects)}")
print(f"vl_subjects: {len(vl_subjects)}")

In [ ]:
#Instantiate ECGDataLoader objects to get a torch.utils.data.DataLoaders for train and validation sets
tr_edl = ECGDataLoader(database=edb, split_subjects=tr_subjects, dynamic_loading=False, batch_size=TRAINING_PARAMS['batch_size'], shuffle=True)
tr_dl = tr_edl.get_dataloader()
vl_edl = ECGDataLoader(database=edb, split_subjects=vl_subjects, dynamic_loading=False, batch_size=TRAINING_PARAMS['batch_size'], shuffle=True)
vl_dl = vl_edl.get_dataloader()

In [ ]:
#Instantiate model components 
text_encoder = TextEncoder(pretrained_model="michiyasunaga/BioLinkBERT-base")
report_decoder = ReportDecoder(decoder_name = 'biogpt')
rd_emb_dim = report_decoder.decoder_input_dimension
ecg_encoder = ECGEncoder(model_architecture="ResNet101", representation_embedding_dim=rd_emb_dim, pretrained_model="resnetPTBXL_weights.pth")
te_emb_dim = text_encoder.embedding_dim
shared_metric_space = SharedMetricSpace(ecg_embedding_dim=ecg_encoder.representation_embedding_dim, text_embedding_dim=text_encoder.embedding_dim, shared_metric_embedding_dim=text_encoder.embedding_dim)
contrastive_learning = ContrastiveLearning()

In [ ]:
tfw = TrainingFramework(ecg_encoder, 
                        text_encoder, 
                        shared_metric_space, 
                        contrastive_learning, 
                        report_decoder, 
                        TRAINING_PARAMS)

In [ ]:
tfw.train(tr_dl, vl_dl)

# eVIP ECG Reporting
This repository is an implementation of an LLM and contrastive learning approach for generating clinical free-text reports from ECG records in the MIMIC-IV dataset. 
* [MIMIC-IV](https://physionet.org/content/mimiciv/3.1/)
* [MIMIC-IV-ECG](https://physionet.org/content/mimic-iv-ecg/1.0/)

## Quick Start

Make sure `MIMIC_IV_DIR = MIMIC_IV_DIR_MINI_SUBSET` in `ECGDataloader.py`. Run cells in `ECGReporting.ipynb` to train a model using the `MIMIC-IV-data-mini-subset` from this repository.

## MIMIC-IV Datasets
The MIMIC-IV datasets used in this project are >100 Gb and contain >3 million files. It is available for download once you become a credentialed user on Physionet. Please email [jonathan.williams@student.unsw.edu.au](mailto:jonathan.williams@student.unsw.edu.au) for instructions. Once downloaded please unzip and make sure it is stored in a folder named `MIMIC-IV-data`. 

This repository contains a synthetic subset of MIMIC-IV in `MIMIC-IV-data-mini-subset` which is only 65Mb and contains only 600 records. This can be used for testing and learning the code. The file structure in `MIMIC-IV-data-mini-subset` mostly resembles the file structure for the full dataset `MIMIC-IV-data`. 

### MIMIC-IV File Structure
- MIMIC-IV-data-mini-subset/
  - mimic-iv-2.2/
    - hosp/
      - admissions.csv
      - d_icd_diagnoses.csv
      - diagnoses_icd.csv
  - mimic-iv-ecg-matched-subset/
    - meta_files/
      - machine_measurements.csv
      - record_list.csv
    - mimic-iv-ecg_complete/
      - all_reports.csv
      - missing_studies.txt
      - files/
        - p{group_id}
          - p{subject_id}
            - s{study_id}
              - {study_id}.dat
              - {study_id}.hea
    - mimic-iv-ecg_complete_300x300_images/
      - all_reports.csv
      - missing_studies.txt
      - files/
        - p{group_id}
          - p{subject_id}
            - s{study_id}
              - {study_id}.png

* `mimic-iv-2.2` contains the hosp module from [MIMIC-IV](https://physionet.org/content/mimiciv/3.1/) which provide diagnostic EMR used for generating synthetic clinical text reports
* `mimic-iv-ecg-matched-subset` contains ECG data from [MIMIC-IV-ECG](https://physionet.org/content/mimic-iv-ecg/1.0/)
* `mimic-iv-ecg_complete` contains the ECG DICOM files (.dat and .hea). For the full dataset `MIMIC-IV-data` the `mimic-iv-ecg_complete` folder will contain DICOM files for ALL studies 
* `mimic-iv-ecg_complete_300x300_images` contains a file structure identical to `mimic-iv-ecg_complete` however the DICOM files are replaced by 300x300 png images of the ECG capture
* `meta_files` contain machine_measurements.csv and record_list.csv from [MIMIC-IV-ECG](https://physionet.org/content/mimic-iv-ecg/1.0/). For the `MIMIC-IV-data-mini-subset` these tables have been shortened down to 600 records. 

## ECGDataLoader.py
This script contains classes for generating ECG images, corresponding (synthetic) clinical text reports, and loading data for machine learning models. 

Classes:
1. **ECGDataBase**
  - Manages the ECG dataset and support dynamic data loading
  - Prepares data paths, processes records, and manages configurations
  - Gets passed to `ECGDataPreparation` and `ECGDataLoader`
2. **ECGDataPreparation**
  - Prepares ECG images and synthetic text reports from raw data
  - Saves prepared data to support static loading
3. **ECGDataLoader** 
  - An implementation of a PyTorch `Dataset` and builds a PyTorch `DataLoader` to facilitate training and validation with MIMIC-IV.
  - Supports both dynamic loading (on-the-fly processing) and static loading (pre-processed images).

## MIMIC Files

* **`missing_studies.txt`**: Lists the MIMIC-IV-ECG studies that were not successfully downloaded using WGET during the data acquisition process. These files are flagged for future redownloading. Until then, `missing_studies.txt` is used by the `ECGDataBase` to exclude these studies from processing to avoid errors caused by missing files.  
* **`record_list.csv`**: A master list of all studies that should be present in the dataset, including missing ones. It acts as a comprehensive index for navigating and processing the data.
  - `subject_id`: The ID of the patient associated with the study.  
  - `study_id`: The unique identifier for the ECG study.  
  - `path`: The relative path to the study folder or file within the dataset.
* **`machine_measurements.csv`**: Provides detailed machine-generated measurements for each ECG study, such as heart rate, RR intervals, and waveforms' onset and offset times. This data is used for report generation and analysis. Each row corresponds to an ECG study and includes measurements such as:  
    - `rr_interval`: Average RR interval.  
    - `p_onset`, `p_end`, `qrs_onset`, `qrs_end`, `t_end`: Timings for the P-wave, QRS complex, and T-wave.  
    - `p_axis`, `qrs_axis`, `t_axis`: Electrical axis measurements.  
    - Additional columns for other machine-generated metrics.
* **`admissions.csv`**: Contains EMR data about patient admissions. It is used to link ECG studies with specific hospital admissions and determine the context for diagnoses.  
    - `subject_id`: Patient ID.  
    - `hadm_id`: Hospital admission ID.  
    - `admittime`, `dischtime`: Timestamps indicating the start and end of a hospital admission.  
    - `edregtime`, `edouttime`: Timestamps for emergency department registration and discharge.  
* **`d_icd_diagnoses.csv`**: Maps International Classification of Diseases (ICD) codes to their descriptions. This is used to interpret and generate text reports for diagnoses related to ECG studies.  
    - `icd_code`: The ICD code for a diagnosis (e.g., `I47`, `4279`).  
    - `long_title`: The full descriptive title of the diagnosis (e.g., `Paroxysmal Tachycardia`, `Atrial Fibrillation`).  
* **`diagnoses_icd.csv`**: Provides the mapping of patient admissions to their corresponding ICD codes for diagnoses recorded during their hospital stay. This data helps identify the specific conditions associated with an ECG study.  
    - `subject_id`: Patient ID.  
    - `hadm_id`: Hospital admission ID.  
    - `icd_code`: ICD code for the recorded diagnosis during the admission. 
* `all_reports.csv`: Contains the synthetic free-text reports pre-generated using `ECGDataPreparation.pregen_reports_threading()` for static loading. One report is generated for each study via template strings which combine structured data from `machine_measurements`, `admissions` and `diagnoses_icd`.  


## Contact

For support or inquiries, please contact the repository manager:

- **Email**: [jonathan.williams@student.unsw.edu.au](mailto:jonathan.williams@student.unsw.edu.au)
- **GitHub**: [@Jon-bon-Jono](https://github.com/Jon-bon-Jono)
